In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# 1. Carregar os dados
df_dados = pd.read_excel("Base Leite 2023-2024 Formatada.xlsx")
df_dados

In [ ]:
df_dados.groupby('sistema')['id_fazenda'].nunique()

In [ ]:
df_dados_2023 = df_dados.loc[
    #((df_dados["sistema"] == "compost-barn") | (df_dados["sistema"] == "free-stall")) &
    (df_dados["sistema"] == "confinado-sem-estrutura") &
    (df_dados["ano"] == 2023)
]
df_dados_2023 = df_dados_2023.drop_duplicates()

df_dados_2023_sorted = df_dados_2023.sort_values(by="69_margem_liquida_unitaria_r_litro", ascending=False).reset_index().drop("index", axis=1)

tot_sup = int(len(df_dados_2023_sorted) * 0.25)
tot_mid = len(df_dados_2023_sorted) - 2*tot_sup
tot_inf = tot_sup
tot_sup, tot_mid, tot_inf

df_dados_2023_25sup = df_dados_2023_sorted.iloc[0:tot_sup, :]
df_dados_2023_50mid = df_dados_2023_sorted.iloc[tot_sup:tot_sup+tot_mid, :]
df_dados_2023_25inf = df_dados_2023_sorted.iloc[tot_sup+tot_mid:, :]

In [ ]:
df_dados_2024 = df_dados.loc[
    #((df_dados["sistema"] == "compost-barn") | (df_dados["sistema"] == "free-stall")) &
    (df_dados["sistema"] == "confinado-sem-estrutura") & 
    (df_dados["ano"] == 2024)
]
df_dados_2024 = df_dados_2024.drop_duplicates()

df_dados_2024_sorted = df_dados_2024.sort_values(by="69_margem_liquida_unitaria_r_litro", ascending=False).reset_index().drop("index", axis=1)

tot_sup = int(len(df_dados_2024_sorted) * 0.25)
tot_mid = len(df_dados_2024_sorted) - 2*tot_sup
tot_inf = tot_sup
tot_sup, tot_mid, tot_inf

df_dados_2024_25sup = df_dados_2024_sorted.iloc[0:tot_sup, :]
df_dados_2024_50mid = df_dados_2024_sorted.iloc[tot_sup:tot_sup+tot_mid, :]
df_dados_2024_25inf = df_dados_2024_sorted.iloc[tot_sup+tot_mid:, :]

In [ ]:
# 2. Definir variáveis de entrada (features) e a variável alvo (target)
features = [
    "id_fazenda",
    "7_total_de_vacas_animais_mes",
    "17_vacas_em_lactacao_total_de_vacas_percentual",
    "13_ccs_contagem_de_celulas_somaticas_x1000_celulas_ml",
    "26_producao_total_de_vacas_litros_dia",
    "33_preco_medio_do_leite_r_litro",
    "34_preco_medio_do_concentrado_r_kg"
]
target = "52_custo_total_do_leite_r_litro"

# Tratar possíveis valores nulos nas colunas selecionadas
# data = df_dados_2023_3000[features + [target]].dropna()

train_data = df_dados_2023_25inf
test_data = df_dados_2024_25inf

# Isolar o X (Características) e o y (Alvo)
X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

# 3. Separar o que é dado numérico e o que é dado categórico
categorical_features = ["id_fazenda"]
numeric_features = [
    "7_total_de_vacas_animais_mes",
    "17_vacas_em_lactacao_total_de_vacas_percentual",
    "13_ccs_contagem_de_celulas_somaticas_x1000_celulas_ml",
    "26_producao_total_de_vacas_litros_dia",
    "33_preco_medio_do_leite_r_litro",
    "34_preco_medio_do_concentrado_r_kg"
]

# 4. Criar o pré-processador
# Variáveis numéricas: são padronizadas (StandardScaler)
# Variáveis categóricas: são transformadas em colunas binárias (OneHotEncoder)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# 5. Dicionário com os Modelos de Machine Learning que serão testados
models = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Rede Neural (Rasa - 64 neurônios)": MLPRegressor(
        hidden_layer_sizes=(64,), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Rede Neural (Média - 64x32)": MLPRegressor(
        hidden_layer_sizes=(64, 32), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Rede Neural (Profunda - 128x64x32)": MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Regressão Ridge (Penalizada)": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "Support Vector Regression (SVR)": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "LightGBM (HistGradientBoosting)": HistGradientBoostingRegressor(random_state=42)
}

# 6. Separar dados de treino (80%) e teste (20%)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Lista para salvar os resultados
results = []

# 7. Treinamento e Avaliação
for name, model in models.items():
    # Cria o pipeline que vai rodar o pré-processamento antes de entregar ao modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Treina o modelo
    pipeline.fit(X_train, y_train)
    
    # Previsões nos dados de teste
    y_pred = pipeline.predict(X_test)
    
    # Cálculos de Erros
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Modelo": name,
        "MAE (R$/L)": round(mae, 4),
        "RMSE (R$/L)": round(rmse, 4),
        "R²": round(r2, 4)
    })

# 8. Mostrar a tabela com os resultados finais
results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

In [ ]:
print(df_dados_2024_25inf[features + [target]].describe().to_markdown())